# Project K, codec-loss control: Qwen2.5-Omni-3B

**Why this notebook exists.** Every Project K run so far fed the
models sound that had passed through a lossy codec: Pilot One at
AAC 96k, Pilot Two and the swap files with the swap build recipe.
A lossy codec throws away quiet detail by design. So a fair
objection stands against the whole record: when a model missed
a quiet sound, maybe the file lost it before the model ever had
the chance. This notebook is the control that answers the
objection. It asks the same model the same questions about the
same sounds, twice over: once with the sound inside an MP4,
decoded the way every earlier run decoded it, and once with the
same sound handed over as an uncompressed WAV waveform that
never passed through the codec.

**The Handover Correction rerun (Nov 3, 2026).** This build
supersedes the two-arm battery above for one reason, raised
by the council review of Series Three: the original Arm A
rode the video path (a video element, use_audio_in_video)
while Arm B rode the image + audio path, so the contrast
mixed the codec with the input path. This rerun carries
three arms. Arm A2 is the Arm A file's own soundtrack
decoded back to WAV (the codec's round trip preserved in
the waveform) and fed through the SAME image + audio path
as Arm B: A2 against B isolates the codec, nothing else.
The original Arm A (video path) stays as a third arm: A
against A2 isolates the input path, nothing else. Seven
sounds, each exactly 10 seconds, each in three arms, each
asked two ways, two repeats: 84 asks. Where the text below
describes two arms and 56 asks, this paragraph governs.
Stimulus provenance for every file: provenance.md in the
codec-control folder.

- **The sounds.** A 1000 Hz tone at the reference level (burst
  RMS -20 dBFS); the same tone 20 dB quieter; three short sharp
  clicks at the quiet level; the spoken sentence (the swap
  study's own speech file, so this control ties back to it);
  the quiet tone mixed under the matched vacuum bed; a quiet
  two-second noise burst, the sizzle-shaped case; and true
  silence.
- **Arm A, the file way.** The sound inside an MP4 built with
  the swap recipe: the same still picture, the native AAC
  encoder. The recipe sets no bitrate and the encoder's rate
  follows the content; this battery's speech file lands at
  about 38 kbps, matching the shipped swap speech file on
  the same soundtrack. Arm A reproduces the recipe, which is
  what every earlier run actually rode.
- **Arm B, the raw way.** The same sound as a WAV master. The
  conversation carries an image element (the swap still, the
  same picture Arm A carries) and an audio element naming the
  WAV file. The model's own tools decode that element with the
  same call they use for a video soundtrack, but the samples
  come from the WAV itself. No MP4 is opened for the audio.
- **The forms.** The open question is the swap study's P3,
  word for word. The yes/no question is the Series Two form,
  "Is there the sound of X in this clip? Answer yes or no.",
  with X matched to the sound. Yes/no rows also carry the yes
  score, read from the same generate call, the way the Series
  Two Qwen runner reads it.

**The gates are files, not answers.** There is no sanity
question in this notebook. A model-answer gate once misfired
on a missing file and stopped a healthy run, so here the
gates are the file and feed assertions, run before the
battery over every file: each file is decoded through the
very path its arm will ride, and the runner prints its path,
SHA-256, duration, and per-second RMS, then asserts the
duration is 10.0 seconds within half a second, asserts the
track is not silent unless the sound is declared silence
(and that declared silence really is zeros), and for Arm B
asserts the waveform the feed path returns hashes equal to
a direct decode of the same WAV in the same session. Any
failure stops the run with exit code 7 before a single ask.
The same proof then repeats per ask, before each answer, on
the exact waveform array the processor receives.

**Resume.** The battery skips any ask already present in
`codec2_qwen_results.csv`, so a stopped session can be
continued by running Step 4 again.

**How to read the result** (fixed in advance, from the codec
control plan):

- Caught raw, missed in AAC: the file did it. That sound's
  earlier misses get re-labeled.
- Missed in both: the model did it; the codec objection dies
  for that sound.
- Caught in both: codec loss does not explain that sound.
- Caught in AAC, missed raw: a build bug. Stop and inspect
  before believing anything.

**Where it runs.** On the free Colab account (kadjei1), under
the routing rule: the 3B is a small model, and this is a
control, not a new battery. **How to run it.** Run the cells
from the top, in order. Step 1 sets up and proves the stack.
Step 2 gets the fifteen battery files. Step 3 writes the
runner and runs its check mode (proof, loading candidates,
file assertions). Step 4 runs the battery and saves
`codec2_qwen_results.csv`.

**The one rule.** We claim only what we ran: these sounds,
this model, these two feed paths, this run.

## Step 1 · Setup

The next cell installs the parts the model needs, with fixed version numbers
so the run is the same every time. It then prints the graphics chip it found.

**Before you run it:** the chip must be a T4. In the Colab menu, open
Runtime, then "Change runtime type", and pick the T4 GPU. On the free plan
this is the normal choice. If the cell prints "no GPU found", stop and change
the runtime type first. The model will not fit in plain memory.

The install takes a few minutes. That is normal.

One more setup step, new after run 2: the first code cell sets a memory
option (expandable segments) before torch loads. It lets the chip's memory
manager grow in pieces instead of reserving one big block, which helps when
the model already takes most of the chip. If you rerun cells in an old
session, restart the runtime once first, so the option takes effect from a
clean start.
**Codec control addition.** At the end, the cell proves the
stack it just installed: it prints the imported transformers
version and the file it loaded from, and stops the run if
the version is not the pinned 4.52.4. The Series Two Qwen
notebook proves its stack the same way. This matters here
because the Qwen model runs under this session's own
Python: there is no private environment, so these installed
versions are the run's versions.


In [ ]:
# --- Setup -------------------------------------------------------------------
# First thing in this cell, before torch is loaded: ask the memory manager
# to use growing segments. This must be set BEFORE torch loads, which is
# why it sits above the torch import below. In a fresh runtime it takes
# effect. If you rerun this cell in a session where torch was already
# loaded, restart the runtime once and run from the top.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Fixed versions, so today's run matches tomorrow's run.
# soundfile and librosa are how the model's tools read the sound track
# out of each video file. Without them, the video would arrive silent,
# and this whole test would prove nothing.
import sys, subprocess

PACKAGES = [
    "transformers==4.52.4",
    "accelerate==1.8.1",
    "qwen-omni-utils==0.0.8",
    "soundfile==0.13.1",
    "librosa==0.11.0",
    "av",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES,
               check=False)
print("install step done")

import torch
if torch.cuda.is_available():
    print("GPU found:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("no GPU found. Stop here. Set the runtime type to the T4 GPU,")
    print("then run this cell again.")
# --- Codec control addition: prove the installed stack -------------------
# The model runs under this same session Python (the Qwen
# notebooks have no private environment), so the versions
# installed above are the versions the run uses. Prove them
# here, and stop if the transformers pin is not met.
import transformers
print("transformers version:", transformers.__version__)
print("transformers file:", transformers.__file__)
if transformers.__version__ != "4.52.4":
    raise SystemExit("STOP: transformers is "
                     + transformers.__version__
                     + ", not the pinned 4.52.4. Do not go on.")
print("stack proof passed: transformers 4.52.4, the Series One pin.")


## Step 2 · Get the codec battery files

Everything this run asks about lives in your Google Drive, in
the folder named `codec-control`. The next cell links your
Drive, finds that folder, and copies fifteen files to a plain
work folder here:

- Seven Arm A files, `codec_<sound>_aac.mp4`: the swap still
  picture carrying each sound, built with the swap recipe.
- Seven Arm B masters, `codec_<sound>_raw.wav`: the same
  sounds as uncompressed WAV, 16-bit PCM at 48000 Hz (the
  speech master is stereo, a byte-for-byte copy of the swap
  study's own speech track; the rest are mono).
- `codec_still.png`: the swap still itself, the picture the
  Arm B asks show the model.

The sounds, in battery order: `tone_ref`, `tone_quiet`,
`clicks_quiet`, `speech`, `tone_under_bed`, `burst_quiet`,
`silence`. Levels and hashes for every file are recorded in
the battery's BATTERY.md; the runner re-measures everything
from inside this session in Step 3 before it believes any of
it.

Do not go on until all twenty-two files say "ready".

In [ ]:
# --- Get the codec battery files -------------------------------------------------
import os, shutil, subprocess

SOUNDS = ["tone_ref", "tone_quiet", "clicks_quiet", "speech",
          "tone_under_bed", "burst_quiet", "silence"]
FILES = ([f"codec_{s}_aac.mp4" for s in SOUNDS]
         + [f"codec_{s}_aac_decoded.wav" for s in SOUNDS]
         + [f"codec_{s}_raw.wav" for s in SOUNDS]
         + ["codec_still.png"])
WORK = "/content/clips"
DRIVE_CLIPS = "/content/drive/MyDrive/codec-control"
os.makedirs(WORK, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

SOURCES = [DRIVE_CLIPS, "/content/codec-control"]

def bring(name):
    dest = os.path.join(WORK, name)
    if os.path.exists(dest):
        return True
    for folder in SOURCES:
        src = os.path.join(folder, name)
        if os.path.exists(src):
            shutil.copy(src, dest)
            print("copied", name, "from", folder)
            return True
    return False

def duration_s(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "csv=p=0", path], capture_output=True, text=True)
    try:
        return float(out.stdout.strip())
    except Exception:
        return None

ready = True
for name in FILES:
    bring(name)
    path = os.path.join(WORK, name)
    if os.path.exists(path):
        d = duration_s(path)
        extra = f", {d:.3f} s" if d is not None else ""
        print("ready:  ", name, f"({os.path.getsize(path) / 1e6:.2f} MB{extra})")
    else:
        ready = False
        print("MISSING:", name)
if not ready:
    print()
    print("!" * 72)
    print("STOP. Some files are missing. Check the Drive folder name is")
    print("exactly codec-control, or upload the files by hand, then")
    print("run this cell again. Do not go on to Step 3.")
    print("!" * 72)
    raise SystemExit("STOP: files missing in Step 2")
print()
print("ready: all 7 Arm A MP4s, all 7 decoded WAVs, all 7 Arm B WAVs, and the still")

## Step 3 · Write the runner, then check: proof, candidates, file assertions

The model never runs inside this notebook's cells directly. It
runs in a plain script, `/content/qwen_codec2_run.py`, under
this session's own Python, the Qwen notebooks' pattern. The
first cell below writes that script. The second runs it in
**check mode**.

Check mode does three things, in order:

1. **Proof again.** The versions and files the runner
   imported, with the transformers pin checked a second time:
   exactly 4.52.4, or the run stops. A line also prints the
   yes-score fact: this model's generate wrapper returns the
   thinker's own result when the talker is off, so the door
   exposes its per-step scores on request.
2. **The loading candidates.** The Series One list, unchanged:
   the 3B in 16-bit form (float16) first, the 7B AWQ build
   second, only if the small one fails. The talker is turned
   off on whichever loads. A failed candidate prints its full
   traceback and appends the same words to
   `codec2_qwen_load_log.txt` in the clips folder.
3. **The battery file assertions.** Every one of the fourteen
   battery files is decoded through the very path its arm
   will ride, printed with its SHA-256, duration, and
   per-second RMS, and asserted: 10.0 seconds within half a
   second; sound where sound is declared, true zeros where
   silence is declared; and for the seven WAVs, the waveform
   the feed path returns must hash equal to a direct decode
   of the same WAV. Any failure stops the run here, exit
   code 7, before a single ask.

There is deliberately no sanity question. The gates are the
files, and they are checked on the model's real feed paths,
not on a text-only warm-up.

In [ ]:
# --- Write the codec control runner script --------------------------------------
# The runner holds the proof, the loading candidates, the
# battery file assertions, the strict per-ask feed proof,
# and the 56 codec asks. It is written to a plain file
# here and only ever run under this session's own Python,
# the interpreter Step 1 installed the stack into.
RUNNER_SOURCE = '\n# qwen_codec2_run.py - the codec-loss control runner for\n# Qwen2.5-Omni-3B. Written by project_k_qwen_codec2_colab.ipynb.\n# It runs under the notebook session\'s own Python: the Qwen\n# notebooks install their stack into the session (Step 1), and\n# there is no private environment to run under. Modes: "check"\n# runs the proof, the loading candidates, and the battery file\n# assertions, then stops. "battery" does all of that and then\n# runs the 56 codec asks and saves the CSV.\nimport os\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\nimport sys\n\nMODE = sys.argv[1] if len(sys.argv) > 1 else "check"\nprint("runner mode:", MODE)\nprint("runner python:", sys.version.split()[0], sys.executable)\n\n# --- Proof lines, printed before anything loads --------------------------------\nimport torch, transformers\nprint("torch version:", torch.__version__)\nprint("transformers version:", transformers.__version__)\nprint("transformers file:", transformers.__file__)\nimport qwen_omni_utils\nprint("qwen_omni_utils file:", qwen_omni_utils.__file__)\nif transformers.__version__ != "4.52.4":\n    print("STOP: this script is running on transformers",\n          transformers.__version__, "and needs exactly 4.52.4,")\n    print("the version the Series One Qwen notebooks pin.")\n    sys.exit(3)\nprint("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()\n      else "none")\n\n# The yes-score question, settled from the model\'s own code\n# before the run (transformers 4.52.4,\n# models/qwen2_5_omni/modeling_qwen2_5_omni.py), exactly as the\n# Series Two Qwen runner settled it: generate is a wrapper that\n# returns the thinker\'s own result when the talker is off, and\n# that result carries its per-step scores when asked. Asking\n# the same greedy call for return_dict_in_generate and\n# output_scores changes nothing about the prompt, the feed, or\n# the tokens chosen; it only keeps numbers the call already\n# computes. So yes/no asks record the probability of a yes\n# token at the first generated position.\nYES_SCORE_EXPOSED = True\nprint("yes token score exposed by the generate door:", YES_SCORE_EXPOSED,\n      "(the wrapper returns the thinker\'s own generate result when",\n      "the talker is off, and that result carries its scores)")\n\n# One more wrapper fact this runner respects, from the same\n# source: the thinker\'s max_new_tokens is pre-seeded from the\n# wrapper\'s own thinker_max_new_tokens parameter, and a plain\n# max_new_tokens is silently ignored. Every generate call\n# below passes its cap as thinker_max_new_tokens.\n\n# --- The load log ----------------------------------------------------------------\nimport traceback, datetime\n\nLOAD_LOG_PATHS = ["/content/drive/MyDrive/codec-control/codec2_qwen_load_log.txt",\n                  "/content/codec2_qwen_load_log.txt"]\n\ndef log_load(text):\n    for p in LOAD_LOG_PATHS:\n        try:\n            with open(p, "a") as fh:\n                fh.write(text)\n        except Exception:\n            pass   # that home is not mounted; the printed copy stands\n\ndef log_paths_saved():\n    return [p for p in LOAD_LOG_PATHS if os.path.exists(p)]\n\nlog_load("=" * 70 + "\\n")\nlog_load("Qwen2.5-Omni-3B codec control load log - "\n         + datetime.datetime.now().isoformat(timespec="seconds")\n         + " - runner mode: " + MODE + "\\n")\n\n# --- Load the model: the Series One candidates, unchanged ----------------------\nfrom transformers import (Qwen2_5OmniForConditionalGeneration,\n                          Qwen2_5OmniProcessor)\n\ndef free_memory():\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nmodel = None\nprocessor = None\nMODEL_NAME = None   # repo name of the model that loaded\nLOAD_MODE = None    # the fit that worked\n\nCANDIDATES = [("Qwen/Qwen2.5-Omni-3B", "fp16"),\n              ("Qwen/Qwen2.5-Omni-7B-AWQ", "AWQ weights, fp16 load")]\n\ncandidate_errors = []\nfor repo, mode in CANDIDATES:\n    print("=" * 70)\n    print(f"candidate: {repo} ({mode})")\n    try:\n        m = Qwen2_5OmniForConditionalGeneration.from_pretrained(\n            repo, torch_dtype=torch.float16, device_map="auto")\n        m.disable_talker()  # written answers only; saves memory\n        p = Qwen2_5OmniProcessor.from_pretrained(repo)\n        model, processor = m, p\n        MODEL_NAME, LOAD_MODE = repo, mode\n        print("loaded:", repo)\n        break\n    except Exception:\n        tb = traceback.format_exc()\n        candidate_errors.append((repo, mode, tb))\n        print("this candidate failed. Full traceback:")\n        print(tb)\n        log_load("-" * 70 + "\\n")\n        log_load("candidate failed: " + repo + " (" + mode + ")\\n")\n        log_load(tb)\n        model = None\n        processor = None\n        free_memory()\n\nprint()\nif model is None:\n    print("!" * 70)\n    print("BOTH CANDIDATES FAILED. No model loaded.")\n    saved = log_paths_saved()\n    if saved:\n        print("The full traceback of every candidate is saved, word")\n        print("for word, in:")\n        for p in saved:\n            print("   ", p)\n    else:\n        print("The load log could not be written anywhere; the full")\n        print("tracebacks are in the output above, and nowhere else.")\n    print("Do not go on. Show this whole output to Muse.")\n    print("!" * 70)\n    sys.exit(4)\nprint("MODEL FOR THIS RUN:", MODEL_NAME, "in", LOAD_MODE, "form")\nlog_load("loaded: " + MODEL_NAME + " in " + LOAD_MODE + " form\\n")\nprint("environment: Colab session python (Step 1 installs),",\n      "transformers 4.52.4")\nif torch.cuda.is_available():\n    used = torch.cuda.memory_allocated() / 1e9\n    print(f"chip memory already used by the model: {used:.1f} GB")\n\n# The yes token ids, read from the model\'s own tokenizer (the\n# processor\'s): the first token of "Yes" and of " Yes". A yes\n# score is the summed probability of those ids at the first\n# generated position.\nYES_TOKEN_IDS = []\nfor _word in ("Yes", " Yes"):\n    _enc = processor.tokenizer.encode(_word, add_special_tokens=False)\n    if _enc:\n        YES_TOKEN_IDS.append(int(_enc[0]))\nYES_TOKEN_IDS = sorted(set(YES_TOKEN_IDS))\nprint("yes token ids for scoring:", YES_TOKEN_IDS,\n      [processor.tokenizer.decode([i]) for i in YES_TOKEN_IDS])\n\n# --- Feed proof machinery ---------------------------------------------------------\n# The audit rule, as in Series Two: the proof is printed from\n# inside the session, per ask, before the answer is taken, and\n# it measures the exact decoded waveform array process_mm_info\n# returns for that ask and hands to the processor: duration,\n# sample count, RMS in dB, nonzero share, the same numbers per\n# one-second slice, and a hash. A copy of each decoded track is\n# saved beside the results. The hard-stop thresholds are the\n# Series Two ones: a track that should carry sound decoding\n# missing, empty, short by more than 0.5 s against the file\'s\n# own duration, or all zeros stops the run with exit code 7;\n# declared silence is exempt by name and is itself verified\n# to be zeros. This runner adds the battery\'s own assertion:\n# every battery file\'s own duration must be 10.0 s within\n# 0.5 s, checked in the assertion pass before the battery and\n# again inside prove_feed on every ask.\nimport csv, hashlib, json, math, re, subprocess\nimport numpy as np\nimport librosa\nimport soundfile as sf\nfrom qwen_omni_utils import process_mm_info\n\nCLIPS = "/content/clips"\nSTILL = os.path.join(CLIPS, "codec_still.png")\nTRACK_DIRS = ["/content/drive/MyDrive/codec-control/codec2_qwen_extracted_tracks",\n              "/content/codec2_qwen_extracted_tracks"]\nfor d in TRACK_DIRS:\n    try:\n        os.makedirs(d, exist_ok=True)\n    except Exception:\n        pass\n\nclass FeedProofError(RuntimeError):\n    pass\n\ndef rms_db(x):\n    if len(x) == 0:\n        return -120.0\n    v = float(np.sqrt(np.mean(np.square(x, dtype=np.float64))))\n    return 20.0 * math.log10(v) if v > 0 else -120.0\n\ndef media_duration(path):\n    out = subprocess.run(\n        ["ffprobe", "-v", "error", "-show_entries", "format=duration",\n         "-of", "csv=p=0", path], capture_output=True, text=True)\n    try:\n        return float(out.stdout.strip())\n    except Exception:\n        return None\n\ndef file_sha256(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as fh:\n        for block in iter(lambda: fh.read(1 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\ndef waveform_sha256(audio_np):\n    arr = np.ascontiguousarray(np.asarray(audio_np, dtype=np.float32))\n    return hashlib.sha256(arr.tobytes()).hexdigest()\n\ndef wav_canonical(path):\n    # The WAV file\'s own decoded samples, decoded here directly\n    # from the WAV (librosa at 16 kHz mono, the same decode the\n    # model\'s tools use), independent of any conversation. Arm B\n    # asserts the waveform the feed path returns hashes equal\n    # to this. Also returns the native-rate decode (soundfile,\n    # downmixed to mono) so the file proof can tie the 16 kHz\n    # waveform back to the samples stored in the file.\n    canon, sr = librosa.load(path, sr=16000, mono=True)\n    canon = np.asarray(canon, dtype=np.float32)\n    native, nsr = sf.read(path, dtype="float32", always_2d=True)\n    native_mono = native.mean(axis=1)\n    return canon, sr, native_mono, nsr\n\nFEED_AAC = (\n    "Arm A, the file way, through the Series One ask path, "\n    "unchanged: the conversation carries the video file "\n    "(fps=1, max_pixels=100352, min_pixels=3136) and the "\n    "question; qwen-omni-utils process_mm_info with "\n    "use_audio_in_video=True checks the file carries an audio "\n    "stream (PyAV) and decodes its AAC soundtrack with librosa "\n    "at 16 kHz mono, and samples the frames in-process; the "\n    "processor builds the model\'s inputs; model.generate is "\n    "called with use_audio_in_video=True, greedy "\n    "(do_sample=False), the token cap passed as "\n    "thinker_max_new_tokens, and only the newly written "\n    "tokens are decoded; frames fed this ask: {frames}; at "\n    "most {tokens} new tokens; the proof measures audios[0], "\n    "the exact decoded waveform array this same "\n    "process_mm_info call handed to the processor")\nFEED_RAW = (\n    "Arm B, the raw way: the conversation carries an image "\n    "element naming codec_still.png (the swap still, the same "\n    "picture the Arm A files carry) and an audio element "\n    "naming the WAV file, and no video element; "\n    "qwen-omni-utils process_mm_info decodes the audio element "\n    "with the same librosa call at 16 kHz mono it uses for a "\n    "video soundtrack, but the samples come from the WAV file "\n    "itself: no MP4 is opened for the audio and no AAC decode "\n    "stands between the master and the model; the processor "\n    "is called with text, audio, and images (no videos, no "\n    "use_audio_in_video) and model.generate runs greedy "\n    "(do_sample=False), the token cap passed as "\n    "thinker_max_new_tokens, only the newly written tokens "\n    "decoded; at most {tokens} new tokens; the proof measures "\n    "audios[0], the exact decoded waveform array this same "\n    "process_mm_info call handed to the processor, and its "\n    "hash was asserted equal to a direct decode of the same "\n    "WAV before the answer was taken")\n\ndef yes_probability(scores):\n    if not scores:\n        return None\n    logits = scores[0][0].float()\n    probs = torch.softmax(logits, dim=-1)\n    total = 0.0\n    for i in YES_TOKEN_IDS:\n        total += float(probs[i].item())\n    return total\n\ndef prove_feed(path, audios, frames_fed, arm, expected_sound,\n               expected_silence):\n    # Measure the waveform array the feed path decoded for this\n    # ask. Raises FeedProofError on any failed check; returns\n    # the proof dict on success.\n    if audios is None or len(audios) == 0 or audios[0] is None:\n        raise FeedProofError(\n            f"the feed path returned no audio for {path}")\n    audio_np = np.asarray(audios[0], dtype=np.float32)\n    if audio_np.ndim > 1:\n        audio_np = audio_np[:, 0]\n    sr = 16000   # the loader\'s own rate, by construction\n    src_dur = media_duration(path)\n    if src_dur is None:\n        raise FeedProofError(\n            f"could not read media duration for {path}")\n    if abs(src_dur - 10.0) > 0.5:\n        raise FeedProofError(\n            f"battery file duration is not 10.0 s for {path}: "\n            f"{src_dur:.3f}s")\n    track_dur = len(audio_np) / sr if sr else 0.0\n    if len(audio_np) == 0:\n        raise FeedProofError(f"decoded track is empty for {path}")\n    if track_dur < src_dur - 0.5:\n        raise FeedProofError(\n            f"decoded track is short for {path}: {track_dur:.3f}s "\n            f"of track for a {src_dur:.3f}s file")\n    overall_rms = round(rms_db(audio_np), 2)\n    nonzero_share = float(np.count_nonzero(audio_np) / len(audio_np))\n    unit_count = int(math.ceil(len(audio_np) / sr))\n    if unit_count <= 0:\n        unit_count = 1\n    per_unit_rms = []\n    per_unit_padding = []\n    zero_chunks = 0\n    for i in range(unit_count):\n        seg = audio_np[i * sr:(i + 1) * sr]\n        per_unit_rms.append(round(rms_db(seg), 2))\n        per_unit_padding.append(int(sr - len(seg)) if len(seg) < sr\n                                else 0)\n        if len(seg) and not np.any(seg):\n            zero_chunks += 1\n    if expected_sound and (nonzero_share == 0.0 or overall_rms <= -119.5\n                           or zero_chunks == unit_count):\n        raise FeedProofError(\n            f"track that should carry sound decoded silent for "\n            f"{path}: rms={overall_rms} dB, "\n            f"nonzero share={nonzero_share}")\n    if expected_silence and nonzero_share > 0.001:\n        raise FeedProofError(\n            f"declared silence track is not silent for {path}: "\n            f"nonzero share={nonzero_share}")\n    track_hash = waveform_sha256(audio_np)\n    track_name = (re.sub(r"[^A-Za-z0-9]+", "_", os.path.basename(path))\n                  + "__" + track_hash[:12] + ".wav")\n    track_paths = []\n    for d in TRACK_DIRS:\n        try:\n            tp = os.path.join(d, track_name)\n            if not os.path.exists(tp):\n                sf.write(tp, audio_np, sr, subtype="PCM_16")\n            track_paths.append(tp)\n        except Exception:\n            pass\n    proof = {\n        "file": os.path.basename(path),\n        "source_file_sha256": file_sha256(path),\n        "frame_ffmpeg_rc": ("n/a (frames are decoded in-process "\n                            "by qwen-omni-utils via decord or "\n                            "torchvision; this path makes no "\n                            "ffmpeg call)"),\n        "audio_ffmpeg_rc": ("n/a (the soundtrack is decoded "\n                            "in-process by qwen-omni-utils via "\n                            "librosa; this path makes no ffmpeg "\n                            "call; ffprobe is used only to read "\n                            "the file\'s own duration for the "\n                            "short-track check)"),\n        "media_duration_s": round(src_dur, 3),\n        "track_duration_s": round(track_dur, 3),\n        "sample_rate": int(sr),\n        "sample_count": int(len(audio_np)),\n        "track_rms_db": overall_rms,\n        "track_nonzero_share": round(nonzero_share, 6),\n        "unit_count": int(unit_count),\n        "per_unit_rms_db": per_unit_rms,\n        "per_unit_padding_samples": per_unit_padding,\n        "padding_samples_total": int(sum(per_unit_padding)),\n        "zero_chunk_count": int(zero_chunks),\n        "track_sha256": track_hash,\n        "track_copy_paths": track_paths,\n        "expected_sound": bool(expected_sound),\n        "expected_silence": bool(expected_silence),\n        "frames_fed": int(frames_fed),\n        "arm": arm,\n        "unit_note": ("unit columns count 1 s slices of the "\n                      "decoded waveform handed to the "\n                      "processor; per-unit padding is the "\n                      "final slice\'s shortfall against a full "\n                      "second, proof bookkeeping only"),\n    }\n    return proof\n\ndef conversation_for(job, prompt):\n    path = os.path.join(CLIPS, job["file"])\n    if job["arm"] in ("raw_wav", "aac_wav"):\n        return [{\n            "role": "user",\n            "content": [\n                {"type": "image", "image": STILL},\n                {"type": "audio", "audio": path},\n                {"type": "text", "text": prompt},\n            ],\n        }]\n    return [{\n        "role": "user",\n        "content": [\n            {"type": "video", "video": path, "fps": 1,\n             "max_pixels": 100352, "min_pixels": 3136},\n            {"type": "text", "text": prompt},\n        ],\n    }]\n\ndef decode_for(job, prompt):\n    # Run the feed path for this job without asking anything:\n    # process_mm_info on the very conversation the ask will\n    # feed, then the proof. Shared by the assertion pass and\n    # by ask(), so what is asserted is what is asked on.\n    path = os.path.join(CLIPS, job["file"])\n    if not os.path.exists(path):\n        raise FeedProofError(f"feed file missing: {path}")\n    conversation = conversation_for(job, prompt)\n    if job["arm"] in ("raw_wav", "aac_wav"):\n        audios, images, videos = process_mm_info(\n            conversation, use_audio_in_video=False)\n        frames_fed = len(images) if images else 0\n    else:\n        audios, images, videos = process_mm_info(\n            conversation, use_audio_in_video=True)\n        frames_fed = 0\n        if videos is not None and len(videos):\n            try:\n                frames_fed = int(videos[0].shape[0])\n            except Exception:\n                frames_fed = 0\n    proof = prove_feed(path, audios, frames_fed, job["arm"],\n                       job["expected_sound"],\n                       job["expected_silence"])\n    if job["arm"] in ("raw_wav", "aac_wav"):\n        canon, csr, native_mono, nsr = wav_canonical(path)\n        canon_hash = waveform_sha256(canon)\n        if canon_hash != proof["track_sha256"]:\n            raise FeedProofError(\n                f"Arm B waveform hash mismatch for {path}: the "\n                f"feed path returned {proof[\'track_sha256\']}, a "\n                f"direct decode of the WAV gives {canon_hash}")\n        proof["wav_canonical_sha256"] = canon_hash\n        proof["wav_native_sample_rate"] = int(nsr)\n        proof["wav_native_rms_db"] = round(rms_db(native_mono), 2)\n        proof["wav_native_duration_s"] = round(len(native_mono) / nsr, 3)\n    return conversation, audios, images, videos, proof\n\ndef ask(job, prompt, max_tokens):\n    # One question, through the model\'s own processor path.\n    # Arm A rides the Series One with-picture ask unchanged.\n    # Arm B rides the image + audio element form. Either way\n    # the feed proof prints before the answer is taken, and\n    # the generate call is the same greedy call, asked to\n    # return its scores so yes/no asks can record the yes\n    # probability from its first score row.\n    conversation, audios, images, videos, proof = decode_for(\n        job, prompt)\n    want_yes = prompt.rstrip().endswith("Answer yes or no.")\n    if job["arm"] in ("raw_wav", "aac_wav"):\n        proof["feed"] = FEED_RAW.format(tokens=max_tokens)\n        print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n        text = processor.apply_chat_template(\n            conversation, add_generation_prompt=True, tokenize=False)\n        inputs = processor(text=text, audio=audios, images=images,\n                           return_tensors="pt", padding=True)\n        inputs = inputs.to(model.device).to(model.dtype)\n        gen_extra = {}\n    else:\n        proof["feed"] = FEED_AAC.format(\n            frames=proof["frames_fed"], tokens=max_tokens)\n        print("FEED PROOF " + json.dumps(proof, sort_keys=True))\n        text = processor.apply_chat_template(\n            conversation, add_generation_prompt=True, tokenize=False)\n        inputs = processor(text=text, audio=audios, images=images,\n                           videos=videos, return_tensors="pt",\n                           padding=True, use_audio_in_video=True)\n        inputs = inputs.to(model.device).to(model.dtype)\n        gen_extra = {"use_audio_in_video": True}\n    with torch.no_grad():\n        out = model.generate(**inputs,\n                             thinker_max_new_tokens=max_tokens,\n                             do_sample=False,\n                             return_dict_in_generate=True,\n                             output_scores=True, **gen_extra)\n    if hasattr(out, "sequences"):\n        seqs = out.sequences\n        scores = getattr(out, "scores", None)\n    elif isinstance(out, tuple):\n        seqs = out[0]\n        scores = None\n    else:\n        seqs = out\n        scores = None\n    new_tokens = seqs[:, inputs.input_ids.shape[1]:]\n    decoded = processor.batch_decode(\n        new_tokens, skip_special_tokens=True,\n        clean_up_tokenization_spaces=False)[0].strip()\n    yp = yes_probability(scores) if want_yes else None\n    del audios, images, videos, inputs, out, seqs, new_tokens\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return decoded, proof, yp\n\n# --- The battery -------------------------------------------------------------------\n# The job list was built and counted by the notebook\'s builder\n# (7 sounds x 3 arms x 2 forms x 2 repeats = 84 asks) and is\n# injected here as a literal. The open prompt is the swap\n# study\'s P3, pulled from pilot2/swap/run_swap.py at build\n# time; the yes/no prompt is the Series Two template with the\n# codec targets.\nJOBS = [({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_ref\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_ref\', \'file\': \'codec_tone_ref_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_quiet\', \'file\': \'codec_tone_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'clicks_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'clicks_quiet\', \'file\': \'codec_clicks_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'clicking\', \'Is there the sound of clicking in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'speech\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'speech\', \'file\': \'codec_speech_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'speech\', \'Is there the sound of speech in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'tone_under_bed\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'tone_under_bed\', \'file\': \'codec_tone_under_bed_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'open\', \'open_p3\', \'burst_quiet\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'burst_quiet\', \'file\': \'codec_burst_quiet_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': True, \'expected_silence\': False}, \'yes_no\', \'matching_yes_no\', \'sizzling\', \'Is there the sound of sizzling in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac.mp4\', \'arm\': \'aac_mp4\', \'version\': \'codec_arm_a\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_aac_decoded.wav\', \'arm\': \'aac_wav\', \'version\': \'codec_arm_a2\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 1, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'open\', \'open_p3\', \'silence\', "What sounds can you hear in this clip? Describe only what is audible, without guessing from the pictures. If the audio is silent, say \'silent\'.", 2, 300), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 1, 16), ({\'test\': \'codec_control\', \'sound\': \'silence\', \'file\': \'codec_silence_raw.wav\', \'arm\': \'raw_wav\', \'version\': \'codec_arm_b\', \'expected_sound\': False, \'expected_silence\': True}, \'yes_no\', \'matching_yes_no\', \'a tone\', \'Is there the sound of a tone in this clip? Answer yes or no.\', 2, 16)]\n\nprint(f"codec battery: {len(JOBS)} asks "\n      f"(7 sounds x 3 arms x 2 forms x 2 repeats)")\n\nFIELDS = ["test", "ask_id", "sound", "file", "arm", "version",\n          "question_form", "question_label", "target", "prompt",\n          "repeat", "target_start_s", "expected_sound",\n          "expected_silence", "frame_ffmpeg_rc", "audio_ffmpeg_rc",\n          "track_duration_s", "sample_rate", "sample_count",\n          "track_rms_db", "track_nonzero_share", "unit_count",\n          "per_unit_rms_db", "per_unit_padding_samples",\n          "padding_samples_total", "zero_chunk_count",\n          "track_sha256", "track_copy_paths", "source_file_sha256",\n          "raw_answer", "yes_score", "yes_score_status", "model",\n          "feed"]\n\nYES_STATUS_EXPOSED = ("exposed: the probability of a yes token at the "\n                      "first generated position, read from the "\n                      "output_scores of the same greedy generate call "\n                      "that produced the answer (the model\'s generate "\n                      "wrapper returns the thinker\'s own result when "\n                      "the talker is off, as in this run); the prompt, "\n                      "the feed, and the decoding are unchanged")\nYES_STATUS_NOT_YESNO = ("not a yes/no ask: this door exposes scores, "\n                        "but this question form has no yes token "\n                        "to score")\nYES_STATUS_MISSING = ("yes/no ask, but no score was recorded for "\n                      "this row; the words or the error in "\n                      "raw_answer are the record")\n\nOUT_PATHS = ["/content/drive/MyDrive/codec-control/codec2_qwen_results.csv",\n             "/content/codec2_qwen_results.csv"]\n\ndef ask_id_of(job, form, target, rep):\n    return (f"{job[\'sound\']}|{job[\'arm\']}|{form}|{target}|{rep}")\n\n# --- Battery file assertions: the only gates ---------------------------------------\n# Before any ask, every battery file is decoded through the\n# very path its arm will ride and asserted. A failure here\n# stops the run with exit code 7 before a single question is\n# asked. There is no model-answer sanity gate in this runner:\n# a past gate misfired on a missing file, so the gates are\n# the files themselves.\ndef assert_battery_files():\n    seen = {}\n    for job, form, label, target, prompt, rep, max_tokens in JOBS:\n        key = (job["file"], job["arm"])\n        if key in seen:\n            continue\n        seen[key] = True\n        path = os.path.join(CLIPS, job["file"])\n        print("-" * 70)\n        print(f"FILE ASSERT {job[\'arm\']}: {job[\'file\']}")\n        try:\n            conversation, audios, images, videos, proof = decode_for(\n                job, prompt)\n        except FeedProofError as e:\n            print()\n            print("!" * 70)\n            print("BATTERY FILE ASSERTION FAILED:")\n            print(str(e))\n            print("No ask has been taken. Fix the file set and run")\n            print("again. Show this whole output to Muse.")\n            print("!" * 70)\n            sys.exit(7)\n        print("FILE PROOF " + json.dumps(proof, sort_keys=True))\n        del audios, images, videos\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    print()\n    print(f"file assertions passed for {len(seen)} files "\n          f"(7 sounds x 3 arms).")\n\nassert_battery_files()\n\nif MODE == "check":\n    print()\n    print("check mode done: proof, candidates, and file assertions")\n    print("all passed. The battery was not run. Run Step 4 for the")\n    print("84 asks.")\n    sys.exit(0)\n\n# --- Resume: load any rows already saved --------------------------------------------\nrows = []\ndone_keys = set()\nfor p in OUT_PATHS:\n    if os.path.exists(p):\n        try:\n            with open(p, newline="") as fh:\n                rows = list(csv.DictReader(fh))\n            print(f"resume: loaded {len(rows)} existing rows from {p}")\n            break\n        except Exception as e:\n            print("resume: could not read", p, "-", str(e)[:120])\n            rows = []\nfor r in rows:\n    done_keys.add((r.get("sound", ""), r.get("arm", ""),\n                   r.get("question_form", ""), r.get("target", ""),\n                   str(r.get("repeat", ""))))\nfailed = 0\n\ndef save_rows():\n    for out_path in OUT_PATHS:\n        try:\n            with open(out_path, "w", newline="") as fh:\n                w = csv.DictWriter(fh, fieldnames=FIELDS)\n                w.writeheader()\n                w.writerows(rows)\n            print("saved:", out_path)\n        except Exception as e:\n            print("could not save", out_path, "-", str(e)[:120])\n\nfor job, form, label, target, prompt, rep, max_tokens in JOBS:\n    key = (job["sound"], job["arm"], form, target, str(rep))\n    aid = ask_id_of(job, form, target, rep)\n    if key in done_keys:\n        print("=" * 70)\n        print(f"SKIP (already in the CSV): {aid}")\n        continue\n    print("=" * 70)\n    print(f"codec_control - sound={job[\'sound\']} arm={job[\'arm\']} "\n          f"version={job[\'version\']} form={form}/{label} "\n          f"target={target} repeat={rep}")\n    want_yes = prompt.rstrip().endswith("Answer yes or no.")\n    raw_text = ""\n    error = ""\n    proof = None\n    yp = None\n    try:\n        raw, proof, yp = ask(job, prompt, max_tokens)\n        print(f"ASK {aid} arm={job[\'arm\']} file={job[\'file\']} "\n              f"waveform_rms_db={proof[\'track_rms_db\']}")\n        if not raw.strip():\n            error = "ERROR: empty answer returned"\n            failed += 1\n            print("WARNING: this ask came back EMPTY. Recorded as an error,")\n            print("never as silence.")\n            raw_text = error\n        else:\n            raw_text = raw\n            print("RAW ANSWER:")\n            print(raw)\n    except FeedProofError as e:\n        print()\n        print("!" * 70)\n        print("FEED PROOF HARD STOP. The feed for this ask failed its proof:")\n        print(str(e))\n        print("No answer was taken for this ask, and the run stops here.")\n        print("Rows completed before this ask are saved now. Show this")\n        print("whole output to Muse.")\n        print("!" * 70)\n        save_rows()\n        sys.exit(7)\n    except Exception as e:\n        error = "ERROR: " + traceback.format_exc()\n        failed += 1\n        print(f"WARNING: this ask FAILED with an error. The full error is")\n        print("saved in the CSV column raw_answer.")\n        raw_text = error\n        proof = proof or {}\n    p = proof or {}\n    if want_yes:\n        if yp is not None:\n            ys, yss = f"{yp:.4f}", YES_STATUS_EXPOSED\n        else:\n            ys, yss = "", YES_STATUS_MISSING\n    else:\n        ys, yss = "n/a", YES_STATUS_NOT_YESNO\n    rows.append({\n        "test": job["test"], "ask_id": aid, "sound": job["sound"],\n        "file": job["file"], "arm": job["arm"],\n        "version": job["version"], "question_form": form,\n        "question_label": label, "target": target, "prompt": prompt,\n        "repeat": rep, "target_start_s": job.get("target_start_s", ""),\n        "expected_sound": job["expected_sound"],\n        "expected_silence": job["expected_silence"],\n        "frame_ffmpeg_rc": p.get("frame_ffmpeg_rc", ""),\n        "audio_ffmpeg_rc": p.get("audio_ffmpeg_rc", ""),\n        "track_duration_s": p.get("track_duration_s", ""),\n        "sample_rate": p.get("sample_rate", ""),\n        "sample_count": p.get("sample_count", ""),\n        "track_rms_db": p.get("track_rms_db", ""),\n        "track_nonzero_share": p.get("track_nonzero_share", ""),\n        "unit_count": p.get("unit_count", ""),\n        "per_unit_rms_db": json.dumps(p.get("per_unit_rms_db", "")),\n        "per_unit_padding_samples": json.dumps(\n            p.get("per_unit_padding_samples", "")),\n        "padding_samples_total": p.get("padding_samples_total", ""),\n        "zero_chunk_count": p.get("zero_chunk_count", ""),\n        "track_sha256": p.get("track_sha256", ""),\n        "track_copy_paths": json.dumps(p.get("track_copy_paths", "")),\n        "source_file_sha256": p.get("source_file_sha256", ""),\n        "raw_answer": raw_text,\n        "yes_score": ys,\n        "yes_score_status": yss,\n        "model": (f"{MODEL_NAME} ({LOAD_MODE}) "\n                  "[session python, transformers 4.52.4]"),\n        "feed": p.get("feed", ""),\n    })\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\nif failed:\n    print()\n    print("!" * 70)\n    print(f"RUN PROBLEM: {failed} ask(s) failed or came back empty. Their")\n    print("raw words or full errors are in the CSV column raw_answer.")\n    print("This run is NOT clean evidence until those asks are explained.")\n    print("!" * 70)\nelse:\n    print()\n    print("Clean run: every ask answered, every feed proved in print first.")\n\nsave_rows()\nif failed:\n    sys.exit(6)\n'
with open("/content/qwen_codec2_run.py", "w") as fh:
    fh.write(RUNNER_SOURCE)
print("runner written:", "/content/qwen_codec2_run.py", len(RUNNER_SOURCE), "chars")

In [ ]:
# --- Check mode: proof, candidates, file assertions --------------------------------
# Runs the runner under this session's own Python and streams
# its output here live. A nonzero exit code means the proof, a
# loading candidate, or a battery file assertion failed, and
# the run stops in this cell. The first check run downloads
# the model files once; that takes a while and is normal.
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_codec2_run.py", "check"])
print()
print("check exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE CHECK RUN FAILED. The runner's own output above says")
    print("where: a proof line, a loading candidate, or a battery file")
    print("assertion. Do not run Step 4. Show this output to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: runner check failed")
print("check passed: proof lines right, model loaded, every battery")
print("file asserted on its real feed path. Go on to Step 4.")

## Step 4 · Run the codec battery

The next cell runs the runner in **battery mode**: proof,
candidates, and the file assertions again, then the 56 asks.
Every ask prints its FEED PROOF line before the model's
answer is taken, and an ASK line naming the ask, its arm,
its file, and the RMS of the waveform actually passed.

**How the arms differ, stated plainly.** Arm A rides the
Series One ask path: the conversation carries the MP4, the
tools decode its AAC soundtrack, the processor takes text,
audio, images, and video. Arm B carries an image element
(the swap still) and an audio element (the WAV file); the
tools decode the WAV with the same call, and the processor
takes text, audio, and images only. The picture content is
the same still in both arms. The soundtrack content is the
same sound in both arms. The only difference is what the
sound passed through on its way in: an AAC encode and
decode, or nothing.

**Token caps.** 300 for the open question, 16 for yes/no,
passed through the wrapper's thinker setting, as in Series
Two. Decoding is greedy everywhere.

**Hard stops and warnings.** A feed that fails its proof
stops the whole run on the spot (exit code 7), with the rows
completed so far saved first. An ask that errors or comes
back empty is saved with its full error text in `raw_answer`
and a WARNING on screen, never as silence, and the run ends
with the RUN PROBLEM banner (exit code 6). If the session
dies, run Step 4 again: asks already in
`codec2_qwen_results.csv` are skipped.

Results save to `codec2_qwen_results.csv` in the Drive clips
folder, one row per ask, with the full feed-proof numbers on
every row, in the Series Two schema plus an ask_id column.

In [ ]:
# --- Battery mode: the codec control asks ------------------------------------------
# Same runner, same session Python, streamed live. The runner
# saves the CSV itself, to the Drive clips folder and to
# /content. A nonzero exit code means the pre-flight gates
# failed up front, a feed proof hard-stopped the run, or the
# run finished with broken asks (the banners above say which).
import subprocess, sys
proc = subprocess.run([sys.executable, "/content/qwen_codec2_run.py", "battery"])
print()
print("battery exit code:", proc.returncode)
if proc.returncode != 0:
    print("!" * 72)
    print("STOP. THE BATTERY RUN DID NOT FINISH CLEAN. Read the runner's")
    print("output above: a gate failed, a feed proof hard-stopped the run,")
    print("or some asks are broken and marked in the CSV. Show it to Muse.")
    print("!" * 72)
    raise SystemExit("STOP: battery run not clean")
print("battery done: every ask answered, every feed proved, CSV saved.")

## Closing · How to read the answers

Read the two arms against each other, sound by sound, before
reading sounds against each other. The reading was fixed in
advance:

- **Caught raw, missed in AAC:** the file did it. The codec,
  not the model, lost that sound, and any earlier Project K
  miss on a sound like it has to be re-labeled.
- **Missed in both:** the model did it. The sound arrived
  intact (the Arm B proof says so, hash and all) and was
  still not reported. The codec objection dies for that
  sound.
- **Caught in both:** codec loss does not explain that sound.
- **Caught in AAC, missed raw:** a build bug, by definition
  of the design. Stop and inspect the feed paths before
  believing any row.

Then compare with the Series Two record. The speech arm of
this battery is the swap study's own speech file, so its two
arms here speak directly to the swap results: if speech is
caught raw and missed in AAC, part of the famous speech
denial was the file. If it is missed in both, the denial is
the model's, now on a feed that provably lost nothing.

- **The yes scores.** Words first, scores beside them, as in
  Series Two: a word that flips between arms while the score
  barely moves, or a score that moves while the word does
  not, is itself a finding about where the loss sits.
- **Scope.** Every claim stays capped at this model, these
  sounds, these two feed paths, this date. Arm B proves what
  arrives when nothing is compressed away; it does not prove
  what any deployed system hears, because deployed systems
  ride files, which is Arm A's world, and that is exactly
  why both arms are in the record.

When you finish, the file `codec2_qwen_results.csv` in your
Drive is the record. Send it back, and the codec control gets
written up next to the Series Two runs.